# Analysis: Aggregate Fine Mappings

In [ ]:
library(data.table)
library(tidyverse)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks data.table::transpose()
✖ lubridate::wday() 


Attaching package: 'kableExtra'

The following object is masked from 'package:dplyr':

    group_rows


Attaching package: 'flextable'

The following objects are masked from 'package:kableExtra':

    as_image, footnote

The following object is masked from 'package:purrr':

    compose

# Inputs

`ns_dir` - Path containing the NemaScan output directory. Used to load the BCSQ fine-mapping results

In [ ]:
ns_dir <- "data/processed/20231116_Analysis_NemaScan"


# Outputs

In [ ]:
full_finemapping_out <- "tables/full_fine_mapping.tsv"


# Functions

In [ ]:

# Function to get trait name from the file path
get_traitname <- function(x) {
  # get the name of the file
  fn <- basename(x)

  # remove the extension and file type info
  id <- gsub("_bcsq_genes_inbred.tsv", "", fn)

  # Regular expression to match Roman numerals
  chrom_pattern <- "(?<=_)(I|II|III|IV|V|VI|X)(?=_)"

  # Split the string at the Roman numeral
  parts <- strsplit(id, chrom_pattern, perl = TRUE)

  # Extract the parts before and after the Roman numeral
  trait <- sapply(parts, `[`, 1)
  interval <- sapply(parts, `[`, 2)

  # Remove the last underscore from 'before' and the first underscore from 'after'
  trait <- sub("_$", "", trait)
  interval <- sub("^_", "", interval)

  # Extract the Roman numeral
  chrom <- regmatches(x, regexpr(chrom_pattern, x, perl = TRUE))

  # Return a list of before, after, and roman parts
  list(trait = trait, interval = interval, chrom = chrom)
}

# Function to load and annotate a single bcsq file with trait info
load_bcsq_file <- function(bcsq_file) {
  # Get trait info from filename

  trait_info <- get_traitname(bcsq_file)

  # Load the file
  bcsq_data <- data.table::fread(bcsq_file)

  # Add trait and interval columns
  bcsq_data <- bcsq_data %>%
    dplyr::mutate(
      trait = trait_info$trait,
      interval = trait_info$interval
    )

  return(bcsq_data)
}


# Main

Aggregate all `*_bcsq_genes_inbred` files into a single table with trait annotations.

In [ ]:

# List all BCSQ fine-mapping files
bcsq_inbred <- list.files(
  path = glue::glue("{ns_dir}/INBRED/Fine_Mappings/Data"),
  pattern = "genes_inbred",
  recursive = TRUE,
  full.names = TRUE
)

n_bcsq_inbred <- length(bcsq_inbred)

print(
  glue::glue("Found {n_bcsq_inbred} INBRED fine mapping BCSQ files")
)


Found 40 INBRED fine mapping BCSQ files

Filtered to 40 INBRED length fine mapping BCSQ files

Aggregated 414966 rows from 40 files

Saved aggregated fine-mapping data to tables/full_fine_mapping.tsv